In [1]:
import os
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import sys

import glob
import h5py
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys# less gpu heavy loss, SAM doesnt look good 
import glob

from torch.utils.data import Dataset
from typing import Optional, Callable, List, Dict, Tuple, Any
from tqdm import tqdm
import json
import wandb

torch.cuda.empty_cache()

In [ ]:
# Set visible CUDA devices
os.environ["CUDA_VISIBLE_DEVICES"] = config["gpus"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/transforms")
sys.path.append("/media/ana-caznok/SSD-08/recon-segment/configs")

In [3]:
from fixed_dataset import FixedDataset
from seg_recon_vit3d import SegRecon_ViT_3D
from transforms.factory import transform_factory
from utils.read_yaml import read_yaml
from utils.model_select import model_select
from loss import loss_select

/home/ana-caznok/software/src/miniforge3/envs/agrvai/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
config_path = glob.glob("/media/ana-caznok/SSD-08/recon-segment/configs/*")
config_file = config_path[0].split('/')[-1]

In [5]:
config = read_yaml(config_path[0])

In [6]:
# --------------------- CONFIG ---------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_PATH = config['base_path']
PREPROCESSING = config['preprocessing']  # or None or "downsampled"
BATCH_SIZE = config['train']['batch_size']
NUM_EPOCHS = config['train']['epochs']
LEARNING_RATE = config['train']['lr']
SAVE_PATH = config['fixed_checkpoint_name']
TRANSFORM = config['train']['transform_index']
MODEL_NAME = config['model']

In [7]:
DEVICE

device(type='cuda')

In [ ]:
# ------------------ DATASET + LOADER ------------------
train_dataset = FixedDataset(
    mode="train",
    base_path=BASE_PATH,
    transform=transform_factory(TRANSFORM),  # You can define transforms here
    preprocessing=PREPROCESSINGavg_train_loss
)

val_dataset = FixedDataset(
    mode="val",
    base_path=BASE_PATH,
    transform=transform_factory(TRANSFORM),
    preprocessing=PREPROCESSING
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Loading from hdf files
Number of face masks 234. 

Checking face mask integrity...: 234it [00:00, 889816.08it/s]


initialized 234 images train DatasetRaw with preprocessing: h5py in /media/ana-caznok/SSD-08/icasp_4090/icasp/data/Link_2 with transform 
0: Downsample by a factor of 4
1: RGB2Pseudo_Hyp with: /media/ana-caznok/SSD-08/recon-segment/ and camera D40
2: FourierSpectralTransform(norm=minmax, transform cube=True, device=cuda). Fold: None
Loading from hdf files
Number of face masks 30. 

Checking face mask integrity...: 30it [00:00, 626015.52it/s]

initialized 30 images val DatasetRaw with preprocessing: h5py in /media/ana-caznok/SSD-08/icasp_4090/icasp/data/Link_2 with transform 
0: Downsample by a factor of 4
1: RGB2Pseudo_Hyp with: /media/ana-caznok/SSD-08/recon-segment/ and camera D40
2: FourierSpectralTransform(norm=minmax, transform cube=True, device=cuda). Fold: None


In [9]:
# ------------------ MODEL ------------------
model = model_select(config).to(DEVICE)
criterion = loss_select(config)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

Selected model seg-rec_61: SegRecon_ViT_3D
Selecting loss: MSE


In [10]:
resume_file = None

In [11]:
def config2wandb(config):
    config_wandb = {} 
    for k in config: 
        if (k == 'train') or (k=='valid'): 
            for l in config[k]:
                config_wandb[k+'_'+l] = config[k][l]
        else: 
            config_wandb[k] = config[k]
    return config_wandb

In [12]:
trn_history = []
val_history = []
# ------------------ INITIALIZING WANDB ------------------
name = os.path.basename(config_file).replace(".yaml", '')
run = wandb.init(project="SegRecViT",
                            reinit=True,
                            config = config2wandb(config), 
                            entity= 'rainbow-ai', 
                            notes="Running experiment",
                            name=name)

# ------------------ TRAIN LOOP ------------------
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for x, y, meta in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Training"):
        
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()


    avg_train_loss = train_loss / len(train_loader)
    trn_history.append(avg_train_loss)

    if config['wandb']:
        wandb.log({f"Train Loss {config['loss']}": avg_train_loss})
        wandb.log({"epoch": epoch})

    # ------------------ EVAL LOOP ------------------
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y, meta in tqdm(val_loader, desc="Validation"):
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            output = model(x)
            loss = criterion(output, y)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    val_history.append(avg_val_loss)

    if config['wandb']:
        wandb.log({f"Validation Loss {config['loss']}": avg_val_loss})
        wandb.log({"epoch": epoch})

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

    if avg_val_loss == np.array(val_history).min(): 
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"Best model saved to {SAVE_PATH}")
    
    if epoch%10 ==0: 
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"Best model saved to {SAVE_PATH}")


# ------------------ SAVE MODEL ------------------
torch.save(model.state_dict(), SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: ana_caznok (rainbow-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Validation: 100%|██████████| 8/8 [00:23<00:00,  2.93s/it]


Epoch 1/30: Train Loss = 0.4242, Val Loss = 0.1882
Best model saved to test
Best model saved to test


Validation: 100%|██████████| 8/8 [00:23<00:00,  2.91s/it]


Epoch 2/30: Train Loss = 0.1488, Val Loss = 0.1424
Best model saved to test


...

Epoch 3/30: Train Loss = 0.1429, Val Loss = 0.1423
Best model saved to test


Epoch 4/30 - Training:   8%|▊         | 5/59 [00:16<02:53,  3.21s/it]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7d0b3543fdc0>> (for post_run_cell):


BrokenPipeError: [Errno 32] Broken pipe

In [ ]:
model(x)

In [ ]:
output.size()